## *The API key has been removed from the submission for security reasons. Please configure the environment variable OPENELECTRICITY_API_KEY before execution.

In [13]:
import requests
import pandas as pd

In [14]:
import os
# Set your OpenElectricity API key as an environment variable
# before running the code
# Example:
# os.environ["OPENELECTRICITY_API_KEY"] = "your_api_key" 
os.environ["OPENELECTRICITY_API_KEY"] = "your_api_key" # 临时设置环境变量（不要提交真实key）
API_KEY = os.getenv("OPENELECTRICITY_API_KEY")

## 1. per-facility power generated (5 min interval)

In [15]:
headers = {
    "Authorization": f"Bearer {API_KEY}"
}

url = "https://api.openelectricity.org.au/v4/facilities/"

params = {
    "with_clerk": "true",
    "network_id": "NEM"
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print(response.status_code)

data = response.json()

print(data.keys())

print(str(data)[:1000])

200
dict_keys(['version', 'created_at', 'success', 'data', 'total_records'])
{'version': '4.5.0.dev43', 'created_at': '2026-05-10T23:06:00+10:00', 'success': True, 'data': [{'code': 'ADP', 'code_display': 'ADP', 'name': 'Adelaide Desalination', 'network_id': 'NEM', 'network_region': 'SA1', 'description': '<p>The Adelaide Desalination plant (ADP), formerly known as the Port Stanvac Desalination Plant, is a sea water reverse osmosis desalination plant located in Lonsdale, South Australia which has the capacity to provide the city of Adelaide with up to 50% of its drinking water needs.</p>', 'osm_way_id': '12798948', 'location': {'lat': -35.100751, 'lng': 138.483357}, 'units': [{'code': 'ADPBA1G', 'code_display': '0ADPBA1G', 'fueltech_id': 'battery_discharging', 'status_id': 'operating', 'capacity_registered': 7.76, 'capacity_maximum': 6.15, 'capacity_storage': 12.6, 'data_first_seen': '2021-05-18T10:55:00+10:00', 'data_last_seen': '2026-05-10T22:35:00+10:00', 'dispatch_type': 'GENERATOR'

In [16]:
facilities_df = pd.DataFrame(data["data"])
print(len(facilities_df))

538


In [17]:
print(facilities_df[["code", "name"]].head(10))

       code                   name
0       ADP  Adelaide Desalination
1   ALDGASF                 Aldoga
2   AMCORGR            Amcor Glass
3  ANGASTON               Angaston
4       APS               Anglesea
5     APPIN                  Appin
6      ARWF                 Ararat
7     AVLSF                Avonlie
8  AWABAREF                  Awaba
9    DEIBDL             Bairnsdale


In [18]:
#调试阶段，先选3个code，正式时选5-10个
#power data

headers = {
    "Authorization": f"Bearer {API_KEY}"
}

facility_codes = [
    "APPIN",      # gas
    "DEIBDL",     # gas
    "BARCALDN",   # gas
    "BAYSW",      # coal
    "ANGASTON",   # distillate

    "ADP",        # battery/renewable
    "APS",        # coal_brown
    "ARWF",       # wind
    "AWABAREF",   # bioenergy
    "BANKSPT"     # distillate
]

url = "https://api.openelectricity.org.au/v4/data/facilities/NEM"

params = [
    ("interval", "5m"),
    ("metrics", "power"),
    ("date_start", "2026-05-01"),
    ("date_end", "2026-05-08")
]

# 添加多个 facility_code
for code in facility_codes:
    params.append(("facility_code", code))

response = requests.get(
    url,
    headers=headers,
    params=params
)

print(response.status_code)

data = response.json()

print(data.keys())

200
dict_keys(['version', 'created_at', 'success', 'data'])


In [19]:
print(data["data"][0].keys())

dict_keys(['network_code', 'metric', 'unit', 'interval', 'date_start', 'date_end', 'groupings', 'results', 'network_timezone_offset'])


In [20]:
results = data["data"][0]["results"]

print(type(results))
print(len(results))

<class 'list'>
13


In [21]:
first = results[0]

print(first.keys())

dict_keys(['name', 'date_start', 'date_end', 'columns', 'data'])


In [22]:
print(results[0]["columns"])
print(results[0]["data"][0])

{'unit_code': 'ADPBA1'}
['2026-05-01T00:00:00+10:00', -0.011]


In [23]:
rows = []

for r in results:

    unit_code = r["columns"]["unit_code"]

    for item in r["data"]:

        rows.append({
            "unit_code": unit_code,
            "timestamp": item[0],
            "power": item[1]
        })

power_df = pd.DataFrame(rows)

power_df["timestamp"] = pd.to_datetime(
    power_df["timestamp"],
    utc=True
)

power_df.head()

,unit_code,timestamp,power
0,ADPBA1,2026-04-30 14:00:00+00:00,-0.011
1,ADPBA1,2026-04-30 14:05:00+00:00,-0.021
2,ADPBA1,2026-04-30 14:10:00+00:00,0.106
3,ADPBA1,2026-04-30 14:15:00+00:00,-0.006
4,ADPBA1,2026-04-30 14:20:00+00:00,-0.037


## 2. per-facility CO2 emissions (5 min interval)

In [24]:
fossil_keywords = ["coal", "gas", "diesel", "distillate"]

rows = []

for _, row in facilities_df.iterrows():
    units = row["units"]

    for unit in units:
        fueltech = unit.get("fueltech_id", "")

        if any(k in fueltech for k in fossil_keywords):
            rows.append({
                "facility_code": row["code"],
                "facility_name": row["name"],
                "network_region": row["network_region"],
                "unit_code": unit.get("code"),
                "fueltech_id": fueltech,
                "status_id": unit.get("status_id")
            })

fossil_df = pd.DataFrame(rows)

fossil_df.head(20)

,facility_code,facility_name,network_region,unit_code,fueltech_id,status_id
0,AMCORGR,Amcor Glass,SA1,AMCORGR,distillate,retired
1,ANGASTON,Angaston,SA1,ANGAST1,distillate,operating
2,ANGASTON,Angaston,SA1,ANGAS1,distillate,retired
3,ANGASTON,Angaston,SA1,ANGAS2,distillate,retired
4,APS,Anglesea,VIC1,APS,coal_brown,retired
5,APPIN,Appin,NSW1,APPIN,gas_wcmg,operating
6,AWABAREF,Awaba,NSW1,AWABAREF,bioenergy_biogas,retired
7,DEIBDL,Bairnsdale,VIC1,BDL01,gas_ocgt,operating
8,DEIBDL,Bairnsdale,VIC1,BDL02,gas_ocgt,operating
9,BBASEHOS,Ballarat Hospital,VIC1,BBASEHOS,gas_recip,retired


In [25]:
headers = {
    "Authorization": f"Bearer {API_KEY}"
}

facility_codes = [
    "APPIN",      # gas
    "DEIBDL",     # gas
    "BARCALDN",   # gas
    "BAYSW",      # coal
    "ANGASTON",   # distillate

    "ADP",        # battery/renewable
    "APS",        # coal_brown
    "ARWF",       # wind
    "AWABAREF",   # bioenergy
    "BANKSPT"     # distillate
]

url = "https://api.openelectricity.org.au/v4/data/facilities/NEM"

params = [
    ("interval", "5m"),
    ("metrics", "emissions"),
    ("date_start", "2026-05-01"),
    ("date_end", "2026-05-08")
]

for code in facility_codes:
    params.append(("facility_code", code))

response = requests.get(url, headers=headers, params=params)

print("Status code:", response.status_code)

data = response.json()

if response.status_code != 200 or not data.get("success", False):
    print(data)
else:
    results = data["data"][0]["results"]

    rows = []

    for r in results:
        unit_code = r["columns"].get("unit_code", None)

        for item in r["data"]:
            rows.append({
                "unit_code": unit_code,
                "timestamp": item[0],
                "emissions": item[1]
            })

    emissions_df = pd.DataFrame(rows)

    emissions_df["timestamp"] = pd.to_datetime(
        emissions_df["timestamp"],
        utc=True
    )

    emissions_df = emissions_df.drop_duplicates()
    emissions_df = emissions_df.dropna(subset=["unit_code", "timestamp"])

    print(emissions_df.head())
    print(emissions_df.shape)

Status code: 200
  unit_code                 timestamp  emissions
0    ADPBA1 2026-04-30 14:00:00+00:00        0.0
1    ADPBA1 2026-04-30 14:05:00+00:00        0.0
2    ADPBA1 2026-04-30 14:10:00+00:00        0.0
3    ADPBA1 2026-04-30 14:15:00+00:00        0.0
4    ADPBA1 2026-04-30 14:20:00+00:00        0.0
(24192, 3)


## 3. per-market price and demand (5 min interval)

In [26]:
headers = {
    "Authorization": f"Bearer {API_KEY}"
}

url = "https://api.openelectricity.org.au/v4/market/network/NEM"

params = [
    ("interval", "5m"),
    ("metrics", "price"),
    ("metrics", "demand"),
    ("date_start", "2026-05-01"),
    ("date_end", "2026-05-08"),
    ("primary_grouping", "network_region")
]

response = requests.get(url, headers=headers, params=params)

print("Status code:", response.status_code)

data = response.json()
print(data.keys())
print(str(data)[:1000])

Status code: 200
dict_keys(['version', 'created_at', 'success', 'data'])
{'version': '4.5.0.dev43', 'created_at': '2026-05-10T23:06:02+10:00', 'success': True, 'data': [{'network_code': 'NEM', 'metric': 'price', 'unit': '$/MWh', 'interval': '5m', 'date_start': '2026-05-01T00:00:00+10:00', 'date_end': '2026-05-07T23:55:00+10:00', 'groupings': [], 'results': [{'name': 'price_NSW1', 'date_start': '2026-05-01T00:00:00+10:00', 'date_end': '2026-05-07T23:55:00+10:00', 'columns': {'region': 'NSW1'}, 'data': [['2026-05-01T00:00:00+10:00', 57.51], ['2026-05-01T00:05:00+10:00', 57.51], ['2026-05-01T00:10:00+10:00', 65.89], ['2026-05-01T00:15:00+10:00', 65.22], ['2026-05-01T00:20:00+10:00', 57.43], ['2026-05-01T00:25:00+10:00', 57.51], ['2026-05-01T00:30:00+10:00', 56.06], ['2026-05-01T00:35:00+10:00', 56.06], ['2026-05-01T00:40:00+10:00', 57.51], ['2026-05-01T00:45:00+10:00', 57.51], ['2026-05-01T00:50:00+10:00', 56.06], ['2026-05-01T00:55:00+10:00', 56.06], ['2026-05-01T01:00:00+10:00', 56.06],

In [27]:
for block in data["data"]:
    print("metric:", block["metric"])
    for r in block["results"]:
        print(r["name"], r["columns"])

metric: price
price_NSW1 {'region': 'NSW1'}
price_QLD1 {'region': 'QLD1'}
price_SA1 {'region': 'SA1'}
price_TAS1 {'region': 'TAS1'}
price_VIC1 {'region': 'VIC1'}
price_WEM {'region': 'WEM'}
metric: demand
demand_NSW1 {'region': 'NSW1'}
demand_QLD1 {'region': 'QLD1'}
demand_SA1 {'region': 'SA1'}
demand_TAS1 {'region': 'TAS1'}
demand_VIC1 {'region': 'VIC1'}
demand_WEM {'region': 'WEM'}


In [28]:
rows = []

for block in data["data"]:
    metric = block["metric"]

    for r in block["results"]:
        region = r["columns"].get("region")

        for item in r["data"]:
            rows.append({
                "region": region,
                "timestamp": item[0],
                metric: item[1]
            })

market_long_df = pd.DataFrame(rows)
market_long_df["timestamp"] = pd.to_datetime(
    market_long_df["timestamp"],
    utc=True
)

market_df = (
    market_long_df
    .groupby(["region", "timestamp"], as_index=False)
    .first()
)

market_df.head()

,region,timestamp,price,demand
0,NSW1,2026-04-30 14:00:00+00:00,57.51,7332.50
1,NSW1,2026-04-30 14:05:00+00:00,57.51,7323.78
2,NSW1,2026-04-30 14:10:00+00:00,65.89,7441.25
3,NSW1,2026-04-30 14:15:00+00:00,65.22,7417.92
4,NSW1,2026-04-30 14:20:00+00:00,57.43,7387.76


# Task 2：Data Integration and Materialisation/Caching

In [29]:
# 2.1 merge power and emissions
combined_df = pd.merge(
    power_df,
    emissions_df,
    on=["unit_code", "timestamp"],
    how="outer"
)

In [30]:
# 2.2 cleaning
combined_df["timestamp"] = pd.to_datetime(
    combined_df["timestamp"],
    utc=True
)

combined_df["power"] = pd.to_numeric(
    combined_df["power"],
    errors="coerce"
)

combined_df["emissions"] = pd.to_numeric(
    combined_df["emissions"],
    errors="coerce"
)

combined_df = combined_df.drop_duplicates()

combined_df = combined_df.dropna(
    subset=["unit_code", "timestamp"]
)

combined_df = combined_df.sort_values(
    by=["timestamp", "unit_code"]
).reset_index(drop=True)

combined_df["power"] = combined_df["power"].round(2)  #保留小数点后两位

combined_df["emissions"] = combined_df["emissions"].round(2)

In [31]:
# 2.3 save
combined_df.to_csv(
    "combined_power_emissions.csv",
    index=False
)

print("combined_power_emissions.csv saved")

combined_power_emissions.csv saved


In [32]:
# 2.4 cleaning market and save
market_df["timestamp"] = pd.to_datetime(market_df["timestamp"])

market_df["price"] = pd.to_numeric(market_df["price"], errors="coerce")
market_df["demand"] = pd.to_numeric(market_df["demand"], errors="coerce")

market_df = market_df.drop_duplicates()
market_df = market_df[
    market_df["region"] != "WEM"         #only NEM
]
market_df = market_df.dropna(
    subset=["region", "timestamp"]
)

market_df = market_df.sort_values(
    by=["timestamp", "region"]
).reset_index(drop=True)

market_df["price"] = market_df["price"].round(2)

market_df["demand"] = market_df["demand"].round(2)

market_df.to_csv(
    "market_price_demand.csv",
    index=False
)

print("market_price_demand.csv saved")
print(market_df.head())

market_price_demand.csv saved
  region                 timestamp  price   demand
0   NSW1 2026-04-30 14:00:00+00:00  57.51  7332.50
1   QLD1 2026-04-30 14:00:00+00:00  64.65  6165.20
2    SA1 2026-04-30 14:00:00+00:00  -2.01  1507.69
3   TAS1 2026-04-30 14:00:00+00:00  78.95   944.41
4   VIC1 2026-04-30 14:00:00+00:00   4.55  4568.84
